# RE_206 · És just el model?

**Optativa d'Aprenentatge automàtic · Bloc 2 — Supervisat · Dificultat 3 · durada: 1 o 2 sessions de 2 h**

Treballes en una consultora que **audita algorismes**. Un banc vol fer servir un model per
**preaprovar** una targeta *premium* a les persones que guanyen més de 50.000 $ l'any, i abans
d'engegar-lo ha de demostrar que el model no discrimina. T'han passat les dades del cens amb què
l'entrenarien. Aquest repte no té una resposta tècnica única: hi ha números que has de calcular
bé, i decisions que has de **justificar**.

**La pregunta:** el model encerta i s'equivoca igual per a dones i homes, i per als diferents grups? I si
traiem `sex` i `race` de les dades, el problema desapareix?

## Les dades

- **Font:** *Adult / Census Income* (Becker i Kohavi, 1996; UCI), a OpenML amb `data_id=1590`.
  Llicència CC BY 4.0.
- **Cada fila** és **una persona** del cens dels Estats Units de 1994 (anonimitzada): 48.842 files i
  15 columnes.
- Documentació: <https://archive.ics.uci.edu/dataset/2/adult>

### Diccionari de columnes

| Columna | Què és |
| --- | --- |
| `class` | **objectiu**: si guanyava més de 50.000 $ l'any (`>50K` / `<=50K`) |
| `age` | edat |
| `workclass` | tipus d'ocupador (privat, administració, autònom...) |
| `fnlwgt` | número que calcula l'oficina del cens. Llegeix a la documentació què és abans de decidir si entra al model |
| `education`, `education-num` | nivell d'estudis, en text i en número |
| `marital-status` | estat civil |
| `occupation` | ocupació |
| `relationship` | relació dins de la llar (`Husband`, `Wife`, `Own-child`, `Not-in-family`...) |
| `race` | grup ètnic, amb les categories del cens americà |
| `sex` | sexe (`Male` / `Female`) |
| `capital-gain`, `capital-loss` | guanys i pèrdues de capital, en dòlars |
| `hours-per-week` | hores de feina setmanals |
| `native-country` | país d'origen |

### Com s'han recollit, i què cal tenir present

- Les dades són del 1994 i dels Estats Units. Les categories de `race` i les de `relationship` són
  les del formulari del cens, no les que faríem servir avui.
- Alguns grups tenen molt poques persones.

## Què has d'investigar

Fes l'auditoria. Abans de cap model, mira si les dades ja són desiguals (proporció de `>50K` per
sexe i per grup, i quanta gent hi ha a cada grup). Decideix quines columnes entren al model i per
què. Entrena el model que s'auditarà (una **regressió logística** dins d'un `Pipeline`), i
compara'n la precisió global amb la d'un `DummyClassifier`.

La teoria ha d'aterrar al codi: escriu una funció que, per a cada grup d'una columna, calculi
**des de la seva matriu de confusió** el nombre de persones, la precisió, la **taxa de selecció**,
la **TPR** i la **FPR**, i comprova la TPR de cada grup amb `recall_score`. Aplica-la per sexe i
per grup ètnic. Després, torna a entrenar **sense** `sex` ni `race` i repeteix l'auditoria: la
diferència desapareix?

**Per anar més enllà** (no és obligatori, i no hi ha una sola manera de fer-ho):

- Es pot predir el sexe a partir de les altres columnes? Què vol dir això per a la pregunta de treure `sex`?
- Un llindar diferent per a cada grup que iguali la TPR: quant canvia la precisió global? És una solució justa? Qui hi guanya i qui hi perd?
- `capital-gain` té un valor que es repeteix molt al màxim. Què és? I unes dades del 1994, serveixen avui?

**Pistes per pensar** (no són la resposta):

- Una precisió més alta en un grup no vol dir que el model el tracti millor: mira quina proporció de les persones que sí que guanyen més de 50K troba en cada grup.
- Treure una columna no treu la informació si altres columnes la porten a dins.
- Una mesura calculada sobre una dotzena de persones es mou molt amb un sol cas: posa la mida del grup al costat de cada número.

## Exemples per inspirar-te

Quaderns del curs on es fan servir les eines que et caldran. Mira com es treballa, no hi busquis
la resposta: les dades d'aquest repte són unes altres.

- [EX_00 · caixa d'eines](../02_practica/EX_00_caixa_eines.ipynb) §3, §4 i §6 (nuls, `SimpleImputer`, `OneHotEncoder`, `ColumnTransformer`) i §7 (matriu de confusió, *recall*, `DummyClassifier`).
- [ML_04 · regressió logística](../01_teoria/ML_04_regressio_logistica.ipynb) i el RE_202: els pesos d'una regressió logística.
- [EX_02 · Breast cancer](../02_practica/EX_02_cancer.ipynb), exercici 7, i el RE_203: el llindar de decisió.

## Lliurament

Lliura aquest quadern **executat**, sense errors, amb la teva anàlisi: cel·les de codi i, entre elles, cel·les de text on expliques què has fet i per què. Cada decisió que prenguis (què fas amb un buit, quina columna treus, quin model tries) ha de tenir la seva línia de justificació. Acaba amb una **conclusió** de tres a cinc frases, escrita per tu, que respongui l'encàrrec amb els números que has trobat. Escriu la conclusió com si fos el **paràgraf final de l'informe d'auditoria** per al banc: què has mesurat, què has trobat, i què recomanes.

> **Quan cridar en Xavi:** per a dubtes de **criteri**, no de sintaxi (per a la sintaxi hi ha la
> documentació i els exemples). Per exemple: «Quina mesura d'equitat és la bona: la mateixa taxa de selecció, la mateixa TPR, o totes dues?» «Com escric això en un informe sense que sembli una acusació ni un rentat de cara?»

La cel·la següent carrega les dades. A partir d'aquí el quadern és teu: afegeix-hi totes les
cel·les de codi i de text que necessitis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=1590, as_frame=True).frame
print(df.shape)
df.head()